# ASL video augmentation (30 clips per class)

Generates augmented copies of existing `.mp4` files under `Videos/ASL/<class>/` until each class folder contains **30** videos.

- **Skips** classes that already have ≥ 30 videos.
- **Appends** new files as the next integer names (`11.mp4`, `12.mp4`, …) after your current `0.mp4 …` files.
- **Does not use horizontal flip** (can swap dominant hand in ASL).

**pip note:** Lines starting with `ERROR: pip's dependency resolver…` are usually **warnings** about **other packages already installed** in this Python environment—not OpenCV failing to install. Here the conflict comes from **`numpy` 2.x** vs packages that still expect **`numpy` &lt; 2** (e.g. scipy, langchain, ai-hedge-fund). Keeping NumPy on **1.26–1.x** (below) avoids that on typical setups; for a completely clean slate, use a **virtual environment** only for this course project.

Install dependencies (run once):

```bash
pip install "opencv-python-headless>=4.8" "numpy>=1.26,<2"
```

In [3]:
# Optional: install if needed (numpy < 2 avoids clashes with scipy / langchain in shared envs)
%pip install -q "opencv-python-headless>=4.8" "numpy>=1.26,<2"


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
from __future__ import annotations

import random
from pathlib import Path

import cv2
import numpy as np

# --- config ---
PROJECT_ROOT = Path(".").resolve()
ASL_ROOT = PROJECT_ROOT / "Videos" / "ASL"
TARGET_PER_CLASS = 30
RANDOM_SEED = 42

random.seed(RANDOM_SEED)
RNG = np.random.default_rng(RANDOM_SEED)

## Augmentation helpers

Combines mild **brightness/contrast**, **HSV jitter**, **Gaussian noise**, **small rotation**, and **temporal cropping** (drops a small random prefix/suffix of frames).

In [5]:
def _adjust_brightness_contrast(bgr: np.ndarray, alpha: float, beta: float) -> np.ndarray:
    return np.clip(bgr.astype(np.float32) * alpha + beta, 0, 255).astype(np.uint8)


def _add_gaussian_noise(bgr: np.ndarray, sigma: float) -> np.ndarray:
    noise = RNG.normal(0.0, sigma, bgr.shape).astype(np.float32)
    return np.clip(bgr.astype(np.float32) + noise, 0, 255).astype(np.uint8)


def _hsv_jitter(bgr: np.ndarray, dh: float, ds: float, dv: float) -> np.ndarray:
    hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV).astype(np.float32)
    hsv[:, :, 0] = (hsv[:, :, 0] + dh) % 180
    hsv[:, :, 1] = np.clip(hsv[:, :, 1] * ds, 0, 255)
    hsv[:, :, 2] = np.clip(hsv[:, :, 2] * dv, 0, 255)
    return cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)


def _rotate_small(bgr: np.ndarray, angle_deg: float) -> np.ndarray:
    h, w = bgr.shape[:2]
    center = (w / 2, h / 2)
    M = cv2.getRotationMatrix2D(center, angle_deg, 1.0)
    return cv2.warpAffine(
        bgr,
        M,
        (w, h),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_REFLECT101,
    )


def augment_frame(bgr: np.ndarray) -> np.ndarray:
    """Apply a random composition of spatial / appearance augmentations."""
    out = bgr

    if RNG.random() < 0.95:
        alpha = float(RNG.uniform(0.82, 1.18))
        beta = float(RNG.uniform(-28, 28))
        out = _adjust_brightness_contrast(out, alpha, beta)

    if RNG.random() < 0.6:
        dh = float(RNG.uniform(-4, 4))
        ds = float(RNG.uniform(0.88, 1.12))
        dv = float(RNG.uniform(0.90, 1.10))
        out = _hsv_jitter(out, dh, ds, dv)

    if RNG.random() < 0.55:
        sigma = float(RNG.uniform(2.0, 7.5))
        out = _add_gaussian_noise(out, sigma)

    if RNG.random() < 0.45:
        angle = float(RNG.uniform(-4.5, 4.5))
        out = _rotate_small(out, angle)

    return out


def temporal_crop(frames: list[np.ndarray]) -> list[np.ndarray]:
    n = len(frames)
    if n <= 4:
        return frames
    max_each = max(1, int(0.12 * n))
    drop_start = int(RNG.integers(0, max_each + 1))
    drop_end = int(RNG.integers(0, max_each + 1))
    if n - drop_start - drop_end < 3:
        return frames
    return frames[drop_start : n - drop_end]

In [6]:
def read_video(path: Path) -> tuple[list[np.ndarray], float]:
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open video: {path}")
    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps is None or fps <= 1e-3:
        fps = 25.0
    frames: list[np.ndarray] = []
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        frames.append(frame)
    cap.release()
    if not frames:
        raise RuntimeError(f"No frames read: {path}")
    return frames, float(fps)


def write_video(path: Path, frames: list[np.ndarray], fps: float) -> None:
    h, w = frames[0].shape[:2]
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(str(path), fourcc, fps, (w, h))
    if not writer.isOpened():
        raise RuntimeError(f"Cannot open VideoWriter for {path}")
    for f in frames:
        if f.shape[0] != h or f.shape[1] != w:
            f = cv2.resize(f, (w, h), interpolation=cv2.INTER_LINEAR)
        writer.write(f)
    writer.release()


def list_numeric_mp4s(folder: Path) -> list[Path]:
    paths = sorted(folder.glob("*.mp4"), key=lambda p: int(p.stem))
    return paths


def augment_one_video(src: Path, dst: Path) -> None:
    frames, fps = read_video(src)
    frames = temporal_crop(frames)
    aug_frames = [augment_frame(fr) for fr in frames]
    write_video(dst, aug_frames, fps)

## Run augmentation for every ASL class

Cycles through existing clips in round-robin order so augmentations spread across all originals.

In [7]:
def fill_class_to_target(class_dir: Path, target: int) -> dict:
    """Return summary counts for one gloss folder."""
    mp4s = list_numeric_mp4s(class_dir)
    n = len(mp4s)
    summary = {"class": class_dir.name, "before": n, "generated": 0, "after": n}

    if n >= target:
        summary["after"] = n
        return summary

    if n == 0:
        raise RuntimeError(f"No source videos in {class_dir}")

    next_idx = max(int(p.stem) for p in mp4s) + 1
    sources = mp4s.copy()
    si = 0

    while n < target:
        src = sources[si % len(sources)]
        si += 1
        dst = class_dir / f"{next_idx}.mp4"
        augment_one_video(src, dst)
        next_idx += 1
        n += 1
        summary["generated"] += 1

    summary["after"] = n
    return summary


assert ASL_ROOT.is_dir(), f"Missing folder: {ASL_ROOT}"

class_dirs = sorted([p for p in ASL_ROOT.iterdir() if p.is_dir()], key=lambda p: p.name.lower())
rows: list[dict] = []

for d in class_dirs:
    rows.append(fill_class_to_target(d, TARGET_PER_CLASS))

for r in rows:
    print(
        f"{r['class']:<14} before={r['before']:>2}  +{r['generated']:<2}  -> after={r['after']}"
    )

total_gen = sum(r["generated"] for r in rows)
print("---")
print(f"Classes processed: {len(rows)}  |  Total new videos written: {total_gen}")

africa         before=11  +19  -> after=30
australia      before= 9  +21  -> after=30
basement       before= 8  +22  -> after=30
bathroom       before= 9  +21  -> after=30
behind         before= 8  +22  -> after=30
cafeteria      before= 7  +23  -> after=30
church         before= 5  +25  -> after=30
city           before=11  +19  -> after=30
college        before=11  +19  -> after=30
country        before= 7  +23  -> after=30
east           before= 9  +21  -> after=30
here           before= 8  +22  -> after=30
home           before= 8  +22  -> after=30
hospital       before= 6  +24  -> after=30
japan          before= 7  +23  -> after=30
north          before= 9  +21  -> after=30
office         before= 7  +23  -> after=30
party          before= 9  +21  -> after=30
restaurant     before= 9  +21  -> after=30
room           before= 9  +21  -> after=30
russia         before= 9  +21  -> after=30
school         before=11  +19  -> after=30
south          before=10  +20  -> after=30
west       

## Optional: preview one augmented clip

Uncomment and set `PREVIEW_CLASS` to a folder name under `ASL`.

In [8]:
# PREVIEW_CLASS = "church"
# d = ASL_ROOT / PREVIEW_CLASS
# originals = list_numeric_mp4s(d)
# if originals:
#     tmp = Path("_preview_aug.mp4")
#     augment_one_video(originals[0], tmp)
#     print("Wrote", tmp.resolve())